# Naive Bayes

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/probabilistic-models/03-naive-bayes

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## A spam filter in 25 lines

Multinomial Naive Bayes on a toy corpus — training really is just counting.

In [ ]:
spam = ["win free money now", "free viagra click now", "claim your free prize money",
        "urgent winner click here", "free money waiting claim now"]
ham  = ["meeting moved to monday", "lunch tomorrow with the team", "draft of the report attached",
        "can you review my code", "notes from the morning meeting"]

vocab = sorted(set(" ".join(spam + ham).split()))
V = len(vocab); idx = {w: i for i, w in enumerate(vocab)}

def counts(docs):
    c = np.zeros(V)
    for d in docs:
        for w in d.split(): c[idx[w]] += 1
    return c

ALPHA = 1.0   # Laplace smoothing
c_spam, c_ham = counts(spam), counts(ham)
logp_w_spam = np.log((c_spam + ALPHA) / (c_spam.sum() + ALPHA * V))
logp_w_ham  = np.log((c_ham  + ALPHA) / (c_ham.sum()  + ALPHA * V))
logprior_spam = np.log(len(spam) / (len(spam) + len(ham)))
logprior_ham  = np.log(len(ham)  / (len(spam) + len(ham)))

In [ ]:
def classify(text):
    ls, lh = logprior_spam, logprior_ham
    for w in text.split():
        if w in idx:
            ls += logp_w_spam[idx[w]]; lh += logp_w_ham[idx[w]]
    p_spam = 1 / (1 + np.exp(lh - ls))
    return p_spam

for t in ["free money meeting", "review the report", "claim free prize", "click now to win"]:
    print(f'{t!r:32}  P(spam) = {classify(t):.3f}')

## Which words carry the signal?

Each word votes with its log-likelihood ratio.

In [ ]:
llr = logp_w_spam - logp_w_ham
order = np.argsort(llr)
words = [vocab[i] for i in np.concatenate([order[:7], order[-7:]])]
vals_ = np.concatenate([llr[order[:7]], llr[order[-7:]]])

plt.figure(figsize=(8, 4.5))
plt.barh(words, vals_, color=['#14b8a6' if v < 0 else '#f43f5e' for v in vals_])
plt.xlabel('log P(w|spam) − log P(w|ham)   (votes toward spam →)')
plt.tight_layout(); plt.show()

## Why smoothing is mandatory

In [ ]:
# rebuild WITHOUT smoothing and classify a spammy email containing one ham-only word
logp_ns_spam = np.log(np.where(c_spam > 0, c_spam / c_spam.sum(), 1e-300))
text = "free money prize meeting"   # 'meeting' never appears in spam
ls = logprior_spam + sum(logp_ns_spam[idx[w]] for w in text.split())
print('unsmoothed spam log-score:', ls)
print('-> one unseen word drove the score to -inf territory,')
print('   vetoing three strong spam words. α=1 smoothing fixes this.')

**Try it:** add a third class ('newsletter'), or port this to a real dataset — scikit-learn's `fetch_20newsgroups` with `CountVectorizer` + this exact math reaches ~85% accuracy on 20 classes.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Laplace smoothing

The fix the "Why smoothing is mandatory" section demands: add a pseudo-count $\alpha$ to every word so no likelihood is ever zero:

$$P(w \mid c) = \frac{\text{count}(w, c) + \alpha}{\text{total}(c) + \alpha V}$$

Implement it. The checks confirm unseen words get nonzero probability, $\alpha = 0$ recovers the raw MLE, and the smoothed likelihoods still sum to 1 over the vocabulary.

In [ ]:
def word_likelihood(count, total, vocab_size, alpha=1.0):
    """Laplace-smoothed P(word | class)."""
    # TODO(you): (count + alpha) / (total + alpha * vocab_size)
    return ...

In [ ]:
# Checks — run me
assert word_likelihood(0, 100, 50) > 0, "unseen words must not zero out the whole product"
assert abs(word_likelihood(0, 100, 50) - 1 / 150) < 1e-12, "(0+1)/(100+50)"
assert abs(word_likelihood(10, 100, 50, alpha=0.0) - 0.1) < 1e-12, "alpha=0 recovers the raw MLE"
assert word_likelihood(100, 100, 50) < 1, "a word in every document must still get P<1 -- smoothing leaves room for the unseen case too"

probs = [word_likelihood(c, 10, 3) for c in [3, 7, 0]]
assert abs(sum(probs) - 1) < 1e-12, "smoothed likelihoods still sum to 1 over the vocabulary"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def word_likelihood(count, total, vocab_size, alpha=1.0):
    return (count + alpha) / (total + alpha * vocab_size)
```

</details>

### Exercise 2 — Classify by summing logs

Naive Bayes scores each class with a sum instead of a product (products of small numbers underflow):

$$\hat{c} = \arg\max_c \Big[ \log P(c) + \sum_{w \in \text{doc}} \log P(w \mid c) \Big]$$

Implement the classifier. Note the empty-document check: with no evidence, **the prior decides**.

In [ ]:
def nb_classify(words, log_priors, log_likes):
    """Return the class with the highest log-score.
    log_priors: {class: log P(c)}; log_likes: {class: {word: log P(w|c)}}.
    Unknown words get log(1e-9)."""
    scores = {}
    for cls, lp in log_priors.items():
        # TODO(you): lp plus the sum of log-likelihoods of the doc's words
        # (hint: log_likes[cls].get(w, np.log(1e-9)))
        scores[cls] = ...

    # TODO(you): the argmax class (hint: max(scores, key=scores.get))
    return ...

In [ ]:
# Checks — run me
log_priors = {"spam": np.log(0.4), "ham": np.log(0.6)}
log_likes = {
    "spam": {"free": np.log(0.05), "money": np.log(0.04), "meeting": np.log(0.001)},
    "ham": {"free": np.log(0.005), "money": np.log(0.004), "meeting": np.log(0.03)},
}

assert nb_classify(["free", "money"], log_priors, log_likes) == "spam", "spammy words win"
assert nb_classify(["meeting"], log_priors, log_likes) == "ham", "work words win"
assert nb_classify([], log_priors, log_likes) == "ham", "no evidence -> the prior decides"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def nb_classify(words, log_priors, log_likes):
    scores = {}
    for cls, lp in log_priors.items():
        scores[cls] = lp + sum(log_likes[cls].get(w, np.log(1e-9)) for w in words)
    return max(scores, key=scores.get)
```

</details>

### Extra practice — DML 140: Bernoulli Naive Bayes

[Open-Deep-ML #140](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/140_bernoulli-naive-bayes-classifier)

The spam filter above is **Multinomial** Naive Bayes — it models *how many times* each word shows up. **Bernoulli** Naive Bayes is the other half of the family: features are just 0/1 (did this word/pixel/flag appear at all?), not counts. The per-feature likelihood becomes a two-sided coin flip:

$$P(x_i \mid c) = p_{i,c}^{\,x_i} \, (1 - p_{i,c})^{\,1 - x_i}$$

so *both* $p_{i,c}$ (probability the feature is present) **and** $1 - p_{i,c}$ (probability it's absent) need their own Laplace-smoothed estimate. Skip smoothing on either side and a feature that is always 0 (or always 1) across every training example of a class drives that side's probability to exactly 0 — one $-\infty$ log-term then vetoes the whole class, no matter how much other evidence points to it.

Implement `NaiveBayes` below, matching DML's exact signature: `forward(self, X, y)` trains on binary features `X` and labels `y`; `predict(self, X)` returns predicted labels. Handle the case where training data contains only one class.

In [ ]:
class NaiveBayes:
    def __init__(self, smoothing=1.0):
        self.smoothing = smoothing
        self.classes_ = None
        self.log_priors_ = None
        self.log_prob_1_ = None   # log P(feature=1 | class), one entry per class
        self.log_prob_0_ = None   # log P(feature=0 | class), one entry per class

    def forward(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.classes_, class_counts = np.unique(y, return_counts=True)
        self.log_priors_ = {cls: np.log(n / len(y)) for cls, n in zip(self.classes_, class_counts)}
        self.log_prob_1_, self.log_prob_0_ = {}, {}
        for cls in self.classes_:
            X_cls = X[y == cls]
            # TODO(you): Laplace-smoothed P(feature=1 | class) for every column at once
            # (hint: (X_cls.sum(axis=0) + smoothing) / (X_cls.shape[0] + 2 * smoothing))
            p1 = ...
            self.log_prob_1_[cls] = np.log(p1)
            self.log_prob_0_[cls] = np.log(1 - p1)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        preds = []
        for row in X:
            scores = {}
            for cls in self.classes_:
                # TODO(you): log prior + sum of the per-feature Bernoulli log-likelihood
                # (hint: row * log_prob_1_[cls] + (1 - row) * log_prob_0_[cls], then .sum())
                scores[cls] = ...
            preds.append(max(scores, key=scores.get))
        return np.array(preds)

In [ ]:
# Checks — run me
model = NaiveBayes(smoothing=1.0)
X = np.array([[1, 0, 1], [1, 1, 0], [0, 0, 1], [0, 1, 0], [1, 1, 1]])
y = np.array([1, 1, 0, 0, 1])
model.forward(X, y)
assert model.predict(np.array([[1, 0, 1]]))[0] == 1, "DML 140's own example: this row should classify as 1"

model2 = NaiveBayes(smoothing=1.0)
model2.forward(np.array([[0], [1], [0], [1]]), np.array([0, 1, 0, 1]))
assert np.array_equal(model2.predict(np.array([[0], [1]])), np.array([0, 1])), "single binary feature, perfectly separable"

# Edge case: a feature that is 0 in EVERY training row of a class
zero_model = NaiveBayes(smoothing=1.0)
zero_model.forward(np.array([[0], [0], [0], [0]]), np.array([0, 0, 1, 1]))
p_present = np.exp(zero_model.log_prob_1_[0][0])
assert 0 < p_present < 1, "Laplace smoothing must keep P(feature=1|class) > 0 even when the observed count is 0"
assert np.isfinite(zero_model.log_prob_1_[0][0]), "log-prob must stay finite, never -inf"

# Edge case: a feature that is 1 in EVERY training row of a class
one_model = NaiveBayes(smoothing=1.0)
one_model.forward(np.array([[1], [1], [1], [1]]), np.array([0, 0, 1, 1]))
p_absent = np.exp(one_model.log_prob_0_[0][0])
assert 0 < p_absent < 1, "Laplace smoothing must keep P(feature=0|class) > 0 even when the feature always fires"
assert np.isfinite(one_model.log_prob_0_[0][0]), "log-prob must stay finite, never -inf"

# Edge case: training data contains only one class
single_model = NaiveBayes(smoothing=1.0)
single_model.forward(np.array([[0, 0], [1, 0], [0, 1]]), np.zeros(3))
preds = single_model.predict(np.array([[1, 1], [0, 0]]))
assert np.array_equal(preds, np.zeros(2)), "with only one class seen in training, every prediction falls back to it"

shape_model = NaiveBayes(smoothing=1.0)
X_rand = np.random.randint(0, 2, (100, 5))
y_rand = np.random.choice([0, 1], size=100)
shape_model.forward(X_rand, y_rand)
preds_rand = shape_model.predict(np.random.randint(0, 2, (10, 5)))
assert preds_rand.shape == (10,), "predictions must be one label per row"

print("✅ Extra practice passed")

<details>
<summary>💡 Show solution</summary>

```python
class NaiveBayes:
    def __init__(self, smoothing=1.0):
        self.smoothing = smoothing
        self.classes_ = None
        self.log_priors_ = None
        self.log_prob_1_ = None
        self.log_prob_0_ = None

    def forward(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.classes_, class_counts = np.unique(y, return_counts=True)
        self.log_priors_ = {cls: np.log(n / len(y)) for cls, n in zip(self.classes_, class_counts)}
        self.log_prob_1_, self.log_prob_0_ = {}, {}
        for cls in self.classes_:
            X_cls = X[y == cls]
            p1 = (X_cls.sum(axis=0) + self.smoothing) / (X_cls.shape[0] + 2 * self.smoothing)
            self.log_prob_1_[cls] = np.log(p1)
            self.log_prob_0_[cls] = np.log(1 - p1)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        preds = []
        for row in X:
            scores = {
                cls: self.log_priors_[cls] + (row * self.log_prob_1_[cls] + (1 - row) * self.log_prob_0_[cls]).sum()
                for cls in self.classes_
            }
            preds.append(max(scores, key=scores.get))
        return np.array(preds)
```

</details>